# Pratica 09 — Topic modeling

**Corso:** NLP Classico con Python · **Lezione 09** · **Tempo stimato:** 60-75 minuti
**Notebook di lezione:** `notebooks_clase/clase09_topic_modeling_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- Gli esercizi da 1 a 5 usano un mini-corpus di 8 frasi (calcio e cucina) e cifre che puoi calcolare a mano. La sfida usa l'estratto del Chisciotte (in spagnolo) di `datos/gutenberg/` (in Colab, carica la cartella `datos/`): un testo letterario antico, non lo spagnolo di oggi (*esempio illustrativo*).
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Librerie: `numpy`, `scikit-learn`, `gensim` e `nltk`.

---
## Cella 0 — Setup e dati

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("scikit-learn", "sklearn")
asegurar("gensim", "gensim")
asegurar("nltk", "nltk")
asegurar("spacy", "spacy")
print("Librerie pronte.")

In [ ]:
import os
import re
import math
import random
import subprocess
import sys
from collections import Counter

import numpy as np
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import adjusted_rand_score
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel

import warnings
warnings.filterwarnings("ignore")
random.seed(42)                          # seme fisso
np.random.seed(42)

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

def cargar_stopwords(idioma):
    """Lista di stopword di NLTK (la scarica se manca)."""
    try:
        return set(stopwords.words(idioma))
    except LookupError:
        nltk.download("stopwords", quiet=True)
        return set(stopwords.words(idioma))

ENCAB = re.compile(r"^(Capítulo|CAPITOLO)\b")   # dato-es
CORPORA = {"spa": ("gutenberg/quijote_cap01-14.txt", "spanish"),
           "ita": ("gutenberg/promessi_sposi_cap01-06.txt", "italian")}

def leer_texto(ruta):
    """Legge un estratto: toglie i segni _corsivo_ e le intestazioni di capitolo e restituisce un solo testo."""
    parrafos = [p.strip() for p in open(ruta, encoding="utf-8").read().replace("_", "").split("\n\n") if p.strip()]
    return " ".join(p for p in parrafos if not ENCAB.match(p))

def nombres_propios(texto, minimo=3, umbral=0.8):
    """Euristica: parole che hanno la maiuscola almeno l'80 % delle volte in cui compaiono fuori da inizio frase."""
    con_mayus, total = Counter(), Counter()
    for m in re.finditer(r"\w+", texto):
        if re.search(r"(^|[.!?…:]\s*[«—\"“]?\s*)$", texto[max(0, m.start() - 4):m.start()]):
            continue                                   # inizio di frase: la maiuscola non informa
        w = m.group()
        total[w.lower()] += 1
        con_mayus[w.lower()] += w[0].isupper()
    return {w for w in total if total[w] >= minimo and con_mayus[w] / total[w] >= umbral}

def trozos(tokens, n=300):
    """Divide una lista di token in pseudodocumenti consecutivi di n token (l'ultimo pezzo corto si scarta se ha meno di n/2)."""
    return [tokens[i:i + n] for i in range(0, len(tokens) - n // 2, n)]

def temas_lda(m, n=8):
    """Le n parole più probabili di ogni tema."""
    return [[w for w, _ in m.show_topic(i, n)] for i in range(m.num_topics)]

def coherencia(m, docs, dic, bow, medida="c_v"):
    """Coerenza di gensim (c_v usa una finestra scorrevole sui testi; u_mass usa i conteggi)."""
    if medida == "u_mass":
        return CoherenceModel(model=m, corpus=bow, dictionary=dic, coherence="u_mass").get_coherence()
    return CoherenceModel(model=m, texts=docs, dictionary=dic, coherence=medida, processes=1).get_coherence()

print("Funzioni ausiliarie caricate.")

In [ ]:
# datos-es
# Due temi inventati su un vocabolario di 4 parole (una riga per tema; ogni riga somma 1)
VOCAB = ["gol", "liga", "sal", "receta"]      # in italiano: gol, campionato, sale, ricetta
PHI = np.array([[0.5, 0.3, 0.1, 0.1],          # tema 1 (P(parola | tema 1))
                [0.1, 0.1, 0.4, 0.4]])         # tema 2

# Mini-corpus di 8 frasi: 4 di calcio e 4 di cucina (spagnolo e italiano)
FRASES = {
    "spa": ["el equipo marcó un gol en la liga", "la liga termina y el equipo gana",
            "el delantero marcó otro gol", "el entrenador cambió al delantero del equipo",
            "la receta lleva sal y aceite", "añade sal a la receta del guiso",
            "el guiso lleva aceite y cebolla", "la cebolla y el aceite del guiso"],
    "ita": ["la squadra ha segnato un gol in campionato", "il campionato finisce e la squadra vince",
            "l'attaccante ha segnato un altro gol", "l'allenatore ha cambiato l'attaccante della squadra",
            "la ricetta richiede sale e olio", "aggiungi sale alla ricetta dello stufato",
            "lo stufato richiede olio e cipolla", "la cipolla e l'olio dello stufato"],
}
TEMA_REAL = ["futbol"] * 4 + ["cocina"] * 4
STOP_E = {"spa": {"el", "la", "un", "y", "en", "del", "al", "otro", "a", "lleva", "añade"},
          "ita": {"il", "la", "lo", "un", "e", "in", "ha", "della", "dello", "alla", "altro", "l", "richiede", "aggiungi"}}

In [ ]:
# Matrice TF-IDF del mini-corpus (spagnolo) e SVD troncata a 2 dimensioni, date per l'esercizio 3
_docs = [[w for w in re.findall(r"\w+", f.lower()) if w not in STOP_E["spa"]] for f in FRASES["spa"]]
_tf = TfidfVectorizer(analyzer=lambda d: d)
_X = _tf.fit_transform(_docs)
VOCAB_E = [str(w) for w in _tf.get_feature_names_out()]
svd = TruncatedSVD(n_components=2, random_state=0).fit(_X)
print("TF-IDF:", _X.shape, "| valori singolari:", np.round(svd.singular_values_, 3))

## Esercizio 1 — Miscela di temi: `P(w | d) = Σ θ · P(w | t)`

Completa `p_palabras(theta, phi)`: dato il vettore di proporzioni dei temi `theta` (somma 1) e la matrice `phi` (una riga per tema, una colonna per parola), restituisci il vettore `P(w | d)` di ogni parola del vocabolario. Completa anche `tema_dominante(theta)`: l'indice del tema con la proporzione maggiore.

Prima di programmare, a mano con `PHI` e `θ = (0,2 ; 0,8)`: `P(gol | d) = 0,2·0,5 + 0,8·0,1 = 0,18`.

In [ ]:
def p_palabras(theta, phi):
    return None    # TODO: prodotto di theta per phi (una riga con NumPy)

def tema_dominante(theta):
    return None    # TODO: indice del massimo

# Controllo automatico
_p = p_palabras([0.2, 0.8], PHI)
assert _p is not None and tema_dominante([0.2, 0.8]) is not None, "Completa le righe contrassegnate con TODO"
assert np.allclose(_p, [0.18, 0.14, 0.34, 0.34]), "rivedi il prodotto: P(gol | d) deve essere 0,18"
assert np.isclose(_p.sum(), 1.0) and tema_dominante([0.2, 0.8]) == 1 and tema_dominante([0.6, 0.4]) == 0
assert np.allclose(p_palabras([1, 0], PHI), PHI[0])          # un documento 100 % del tema 1 riproduce il tema
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Pre-trattamento: stopword, nomi propri e lemmi

Completa `preparar_p(tokens, stop, nombres, lemas=None, minimo=3)`: passa a minuscole, e togli le stopword (`stop`), le parole di meno di `minimo` lettere e i nomi propri (`nombres`). Se si dà `lemas` (una lista della stessa dimensione di `tokens`), si usano **i lemmi al posto delle forme**.

Prima di programmare, a mano: con `["El", "cura", "dijo", "que", "el", "barbero", "era", "Sancho"]`, `stop = {"dijo", "que", "era"}` e `nombres = {"sancho"}` resta `["cura", "barbero"]` (`el` ha 2 lettere).

In [ ]:
def preparar_p(tokens, stop, nombres, lemas=None, minimo=3):
    fuente = None      # TODO: i lemmi se dati; altrimenti i token in minuscolo
    return None        # TODO: lista di parole di `fuente` che non siano in stop né in nombres e abbiano almeno `minimo` lettere

# Controllo automatico
FRASE_E = ["El", "cura", "dijo", "que", "el", "barbero", "era", "Sancho"]          # dato-es
STOP_2, NOM_2 = {"dijo", "que", "era"}, {"sancho"}                                  # dato-es
LEM_E = ["el", "cura", "decir", "que", "el", "barbero", "ser", "sancho"]            # dato-es
SIN_FILTRO_E = ["el", "cura", "dijo", "que", "el", "barbero", "era", "sancho"]            # dato-es
_r = preparar_p(FRASE_E, STOP_2, NOM_2)
assert _r is not None, "Completa le righe contrassegnate con TODO"
assert _r == ["cura", "barbero"], "rivedi: minuscole, stopword, nomi e parole corte"
assert preparar_p(FRASE_E, set(), set(), minimo=1) == SIN_FILTRO_E
assert preparar_p(FRASE_E, STOP_2, NOM_2, lemas=LEM_E) == ["cura", "decir", "barbero", "ser"]
print("Esercizio 2 corretto ✔")

## Esercizio 3 — LSA: le dimensioni e il segno arbitrario

Nella cella dei dati si costruisce una matrice TF-IDF del mini-corpus e si adatta `TruncatedSVD` con 2 dimensioni (`svd`). Completa `fijar_signo(v)`: restituisce `v` se la sua componente di maggior valore assoluto è positiva, e `-v` altrimenti (così due adattamenti che differiscono solo per il segno danno lo stesso vettore). Poi completa `palabras_positivas(comp, vocab, n)`: le `n` parole di peso maggiore **dopo aver fissato il segno**.

Verifica a mano: `fijar_signo([0,1; -0,9; 0,3])` = `[-0,1; 0,9; -0,3]`.

In [ ]:
def fijar_signo(v):
    v = np.asarray(v, dtype=float)
    return None    # TODO: v o -v secondo il segno della componente di maggior valore assoluto

def palabras_positivas(comp, vocab, n=3):
    comp = fijar_signo(comp)
    return None    # TODO: le n parole di peso maggiore (usa np.argsort)

# Controllo automatico
assert fijar_signo([0.1, -0.9, 0.3]) is not None, "Completa le righe contrassegnate con TODO"
assert np.allclose(fijar_signo([0.1, -0.9, 0.3]), [-0.1, 0.9, -0.3]) and np.allclose(fijar_signo([-0.1, 0.9, -0.3]), [-0.1, 0.9, -0.3])
assert all(np.allclose(fijar_signo(c), fijar_signo(-c)) for c in svd.components_), "fijar_signo(v) e fijar_signo(-v) devono coincidere"
_top = palabras_positivas(svd.components_[1], VOCAB_E, 3)
assert _top is not None and len(_top) == 3 and len(set(_top)) == 3 and all(w in VOCAB_E for w in _top)
_c = fijar_signo(svd.components_[1])
assert [VOCAB_E.index(w) for w in _top] == list(np.argsort(-_c)[:3]), "devono essere le 3 di peso maggiore, in ordine"
print("Esercizio 3 corretto ✔", _top)

## Esercizio 4 — LDA sui conteggi, con seme fisso

Completa `entrenar(docs, k, semilla=0, no_below=1, no_above=1.0)`: `docs` è una lista di liste di parole. Costruisci un `Dictionary`, filtra con `filter_extremes(no_below, no_above)`, calcola i **conteggi** di ogni documento con `doc2bow` (non TF-IDF) e addestra un `LdaModel` con `num_topics=k`, `random_state=semilla`, `passes=10`, `iterations=50` e `eval_every=None`. Restituisci `(modelo, diccionario, bow)`.

Verifiche a mano: senza filtro, il dizionario ha tante voci quante sono le parole distinte, e i conteggi del documento 0 sommano il suo numero di parole.

In [ ]:
def entrenar(docs, k, semilla=0, no_below=1, no_above=1.0):
    dic = None     # TODO: Dictionary(docs) e poi filter_extremes(no_below=no_below, no_above=no_above)
    bow = None     # TODO: [dic.doc2bow(d) for d in docs]
    modelo = None  # TODO: LdaModel(bow, num_topics=k, id2word=dic, random_state=semilla, passes=10, iterations=50, eval_every=None)
    return modelo, dic, bow

# Controllo automatico
DOCS_E = [[w for w in re.findall(r"\w+", f.lower()) if w not in STOP_E["spa"]] for f in FRASES["spa"]]
_m, _dic, _bow = entrenar(DOCS_E, 2)
assert _m is not None and _dic is not None, "Completa le righe contrassegnate con TODO"
assert len(_dic) == len({w for d in DOCS_E for w in d}), "senza filtro, il dizionario deve avere tutte le parole distinte"
assert sum(c for _, c in _bow[0]) == len(DOCS_E[0]), "i conteggi del documento 0 devono sommare il suo numero di parole"
assert _m.get_topics().shape == (2, len(_dic)) and np.allclose(_m.get_topics().sum(axis=1), 1.0, atol=1e-5)
_theta = np.array([[p for _, p in _m.get_document_topics(b, minimum_probability=0.0)] for b in _bow])
assert _theta.shape == (8, 2) and np.allclose(_theta.sum(axis=1), 1.0, atol=1e-5)
_m2 = entrenar(DOCS_E, 2)[0]
assert np.allclose(_m.get_topics(), _m2.get_topics()), "con lo stesso seme, i temi devono essere identici"
assert entrenar(DOCS_E, 2, semilla=1)[0].num_topics == 2
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Misurare se un modello recupera i temi noti: purezza e ARI

Completa `pureza(y, grupos)`: per ogni gruppo, conta la classe maggioritaria (con `Counter`) e somma; dividi per il totale. Poi calcola l'ARI con `adjusted_rand_score` per l'esempio della cella.

A mano: `y = [a,a,a,b,b,b]` e `grupos = [0,0,1,1,1,1]`. Il gruppo 0 ha {a,a} → 2; il gruppo 1 ha {a,b,b,b} → 3. Purezza = 5/6. I numeri dei gruppi sono arbitrari: rinominarli non cambia la purezza.

In [ ]:
Y_E = ["a", "a", "a", "b", "b", "b"]        # dato-es
G_E = [0, 0, 1, 1, 1, 1]

def pureza(y, grupos):
    total = 0
    for g in sorted(set(grupos)):
        pass       # TODO: somma la frequenza della classe maggioritaria tra i documenti del gruppo g
    return None    # TODO: total diviso per len(y)

ari_e = None       # TODO: ARI tra Y_E e G_E

# Controllo automatico
assert pureza(Y_E, G_E) is not None and ari_e is not None, "Completa le righe contrassegnate con TODO"
assert math.isclose(pureza(Y_E, G_E), 5 / 6), "la purezza di questo esempio è 5/6"
assert pureza(Y_E, [1, 1, 0, 0, 0, 0]) == pureza(Y_E, G_E), "rinominare i gruppi non cambia la purezza"
assert pureza(Y_E, [0, 0, 0, 1, 1, 1]) == 1.0 and pureza(Y_E, [0, 1, 2, 3, 4, 5]) == 1.0      # attenzione: un gruppo per documento dà purezza 1 anche lui
assert math.isclose(ari_e, adjusted_rand_score(Y_E, [1, 1, 0, 0, 0, 0])) and adjusted_rand_score(Y_E, [0, 0, 0, 1, 1, 1]) == 1.0
assert 0 < ari_e < 1
print("Esercizio 5 corretto ✔", round(pureza(Y_E, G_E), 3), round(ari_e, 3))

---
## Sfida finale — I nomi propri e il numero di temi nel Chisciotte

Con `datos/gutenberg/quijote_cap01-14.txt` (spagnolo del Seicento; *esempio illustrativo*): pseudodocumenti di 300 token, con **le tue** funzioni `preparar_p` (esercizio 2) ed `entrenar` (esercizio 4).

1. Prepara `docs_sin` (senza nomi propri) e `docs_con` (con nomi propri), entrambi con le stopword di NLTK.
2. Completa `nombres_en_top(modelo, nombres, n=8)`: quante delle `n` parole principali di tutti i temi sono nomi propri.
3. Addestra LDA con `k = 5`, seme 0, `no_below=3`, `no_above=0.25` su `docs_con` e `docs_sin` e calcola `n_con` e `n_sin`.
4. Per `k` in (3, 4, 5, 6) e semi (0, 1), calcola la coerenza **c_v** media (con `coherencia(modelo, docs_sin, dic, bow)`) e salva in `mejor_k` il `k` con la media maggiore. Ricorda: è una guida rumorosa, non la risposta.

In [ ]:
ruta, nombre_nltk = CORPORA["spa"]
texto = leer_texto(cargar_datos(ruta))
NOMBRES = nombres_propios(texto)
STOP = cargar_stopwords(nombre_nltk)
trozos_q = trozos(re.findall(r"\w+", texto), 300)
docs_sin = None    # TODO: preparar_p su ogni pezzo, togliendo stopword e nomi
docs_con = None    # TODO: preparar_p su ogni pezzo, togliendo solo le stopword

def nombres_en_top(modelo, nombres, n=8):
    return None    # TODO: conta le parole di temas_lda(modelo, n) che sono in nombres

n_con = n_sin = None   # TODO: nomi nel top di LDA (k=5, seme 0, no_below=3, no_above=0.25) con docs_con e con docs_sin

coh_media = {}
for k in (3, 4, 5, 6):
    pass           # TODO: media di c_v con semi 0 e 1 su docs_sin (stessi filtri); salvala in coh_media[k]
mejor_k = None     # TODO: il k con coerenza media maggiore

# Controllo automatico
assert docs_sin is not None and n_con is not None and mejor_k is not None, "Completa le righe contrassegnate con TODO"
assert len(docs_sin) == len(docs_con) > 90 and all(len(a) <= len(b) for a, b in zip(docs_sin, docs_con))
assert n_sin == 0, "senza nomi propri non dovrebbe restarne nessuno tra le parole principali"
assert n_con >= 3, "con i nomi propri dovrebbero comparirne diversi"
assert mejor_k in (3, 4, 5, 6) and set(coh_media) == {3, 4, 5, 6}
print("Sfida finale corretta ✔")
print("n_con =", n_con, "| n_sin =", n_sin, "| c_v media =", {k: round(v, 3) for k, v in coh_media.items()}, "| mejor_k =", mejor_k)

**Le tue conclusioni (scrivi 3-4 frasi):** perché il segno di una dimensione di LSA non significa nulla? Perché LDA si addestra sui conteggi e non su TF-IDF? Quali decisioni di pre-trattamento hanno cambiato i temi del Chisciotte, e perché non puoi confrontare il c_v di due preparazioni diverse né quello di un corpus spagnolo con quello di uno italiano? Chi mette l'etichetta a un tema?

*(La tua risposta qui.)*

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
def p_palabras(theta, phi):
    return np.asarray(theta) @ np.asarray(phi)

def tema_dominante(theta):
    return int(np.argmax(theta))

# Controllo automatico
_p = p_palabras([0.2, 0.8], PHI)
assert _p is not None and tema_dominante([0.2, 0.8]) is not None, "Completa le righe contrassegnate con TODO"
assert np.allclose(_p, [0.18, 0.14, 0.34, 0.34]), "rivedi il prodotto: P(gol | d) deve essere 0,18"
assert np.isclose(_p.sum(), 1.0) and tema_dominante([0.2, 0.8]) == 1 and tema_dominante([0.6, 0.4]) == 0
assert np.allclose(p_palabras([1, 0], PHI), PHI[0])          # un documento 100 % del tema 1 riproduce il tema
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
def preparar_p(tokens, stop, nombres, lemas=None, minimo=3):
    fuente = lemas if lemas is not None else [w.lower() for w in tokens]
    return [w for w in fuente if w not in stop and w not in nombres and len(w) >= minimo]

# Controllo automatico
FRASE_E = ["El", "cura", "dijo", "que", "el", "barbero", "era", "Sancho"]          # dato-es
STOP_2, NOM_2 = {"dijo", "que", "era"}, {"sancho"}                                  # dato-es
LEM_E = ["el", "cura", "decir", "que", "el", "barbero", "ser", "sancho"]            # dato-es
SIN_FILTRO_E = ["el", "cura", "dijo", "que", "el", "barbero", "era", "sancho"]            # dato-es
_r = preparar_p(FRASE_E, STOP_2, NOM_2)
assert _r is not None, "Completa le righe contrassegnate con TODO"
assert _r == ["cura", "barbero"], "rivedi: minuscole, stopword, nomi e parole corte"
assert preparar_p(FRASE_E, set(), set(), minimo=1) == SIN_FILTRO_E
assert preparar_p(FRASE_E, STOP_2, NOM_2, lemas=LEM_E) == ["cura", "decir", "barbero", "ser"]
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
def fijar_signo(v):
    v = np.asarray(v, dtype=float)
    return v if v[np.argmax(np.abs(v))] > 0 else -v

def palabras_positivas(comp, vocab, n=3):
    comp = fijar_signo(comp)
    return [str(vocab[i]) for i in np.argsort(-comp)[:n]]

# Controllo automatico
assert fijar_signo([0.1, -0.9, 0.3]) is not None, "Completa le righe contrassegnate con TODO"
assert np.allclose(fijar_signo([0.1, -0.9, 0.3]), [-0.1, 0.9, -0.3]) and np.allclose(fijar_signo([-0.1, 0.9, -0.3]), [-0.1, 0.9, -0.3])
assert all(np.allclose(fijar_signo(c), fijar_signo(-c)) for c in svd.components_), "fijar_signo(v) e fijar_signo(-v) devono coincidere"
_top = palabras_positivas(svd.components_[1], VOCAB_E, 3)
assert _top is not None and len(_top) == 3 and len(set(_top)) == 3 and all(w in VOCAB_E for w in _top)
_c = fijar_signo(svd.components_[1])
assert [VOCAB_E.index(w) for w in _top] == list(np.argsort(-_c)[:3]), "devono essere le 3 di peso maggiore, in ordine"
print("Esercizio 3 corretto ✔", _top)

### Soluzione dell'esercizio 4

In [ ]:
def entrenar(docs, k, semilla=0, no_below=1, no_above=1.0):
    dic = Dictionary(docs)
    dic.filter_extremes(no_below=no_below, no_above=no_above)
    bow = [dic.doc2bow(d) for d in docs]
    modelo = LdaModel(bow, num_topics=k, id2word=dic, random_state=semilla, passes=10, iterations=50, eval_every=None)
    return modelo, dic, bow

# Controllo automatico
DOCS_E = [[w for w in re.findall(r"\w+", f.lower()) if w not in STOP_E["spa"]] for f in FRASES["spa"]]
_m, _dic, _bow = entrenar(DOCS_E, 2)
assert _m is not None and _dic is not None, "Completa le righe contrassegnate con TODO"
assert len(_dic) == len({w for d in DOCS_E for w in d}), "senza filtro, il dizionario deve avere tutte le parole distinte"
assert sum(c for _, c in _bow[0]) == len(DOCS_E[0]), "i conteggi del documento 0 devono sommare il suo numero di parole"
assert _m.get_topics().shape == (2, len(_dic)) and np.allclose(_m.get_topics().sum(axis=1), 1.0, atol=1e-5)
_theta = np.array([[p for _, p in _m.get_document_topics(b, minimum_probability=0.0)] for b in _bow])
assert _theta.shape == (8, 2) and np.allclose(_theta.sum(axis=1), 1.0, atol=1e-5)
_m2 = entrenar(DOCS_E, 2)[0]
assert np.allclose(_m.get_topics(), _m2.get_topics()), "con lo stesso seme, i temi devono essere identici"
assert entrenar(DOCS_E, 2, semilla=1)[0].num_topics == 2
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
Y_E = ["a", "a", "a", "b", "b", "b"]        # dato-es
G_E = [0, 0, 1, 1, 1, 1]

def pureza(y, grupos):
    total = 0
    for g in sorted(set(grupos)):
        total += max(Counter(yi for yi, gi in zip(y, grupos) if gi == g).values())
    return total / len(y)

ari_e = adjusted_rand_score(Y_E, G_E)

# Controllo automatico
assert pureza(Y_E, G_E) is not None and ari_e is not None, "Completa le righe contrassegnate con TODO"
assert math.isclose(pureza(Y_E, G_E), 5 / 6), "la purezza di questo esempio è 5/6"
assert pureza(Y_E, [1, 1, 0, 0, 0, 0]) == pureza(Y_E, G_E), "rinominare i gruppi non cambia la purezza"
assert pureza(Y_E, [0, 0, 0, 1, 1, 1]) == 1.0 and pureza(Y_E, [0, 1, 2, 3, 4, 5]) == 1.0      # attenzione: un gruppo per documento dà purezza 1 anche lui
assert math.isclose(ari_e, adjusted_rand_score(Y_E, [1, 1, 0, 0, 0, 0])) and adjusted_rand_score(Y_E, [0, 0, 0, 1, 1, 1]) == 1.0
assert 0 < ari_e < 1
print("Esercizio 5 corretto ✔", round(pureza(Y_E, G_E), 3), round(ari_e, 3))

### Soluzione della sfida finale

In [ ]:
ruta, nombre_nltk = CORPORA["spa"]
texto = leer_texto(cargar_datos(ruta))
NOMBRES = nombres_propios(texto)
STOP = cargar_stopwords(nombre_nltk)
trozos_q = trozos(re.findall(r"\w+", texto), 300)
docs_sin = [preparar_p(x, STOP, NOMBRES) for x in trozos_q]
docs_con = [preparar_p(x, STOP, set()) for x in trozos_q]

def nombres_en_top(modelo, nombres, n=8):
    return sum(w in nombres for palabras in temas_lda(modelo, n) for w in palabras)

n_con = nombres_en_top(entrenar(docs_con, 5, 0, 3, 0.25)[0], NOMBRES)
n_sin = nombres_en_top(entrenar(docs_sin, 5, 0, 3, 0.25)[0], NOMBRES)

coh_media = {}
for k in (3, 4, 5, 6):
    valores = []
    for s in (0, 1):
        m, dic, bow = entrenar(docs_sin, k, s, 3, 0.25)
        valores.append(coherencia(m, docs_sin, dic, bow, "c_v"))
    coh_media[k] = float(np.mean(valores))
mejor_k = max(coh_media, key=coh_media.get)

# Controllo automatico
assert docs_sin is not None and n_con is not None and mejor_k is not None, "Completa le righe contrassegnate con TODO"
assert len(docs_sin) == len(docs_con) > 90 and all(len(a) <= len(b) for a, b in zip(docs_sin, docs_con))
assert n_sin == 0, "senza nomi propri non dovrebbe restarne nessuno tra le parole principali"
assert n_con >= 3, "con i nomi propri dovrebbero comparirne diversi"
assert mejor_k in (3, 4, 5, 6) and set(coh_media) == {3, 4, 5, 6}
print("Sfida finale corretta ✔")
print("n_con =", n_con, "| n_sin =", n_sin, "| c_v media =", {k: round(v, 3) for k, v in coh_media.items()}, "| mejor_k =", mejor_k)

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto non ti è rimasto chiaro (tema come distribuzione, LSA e il suo segno, LDA e i conteggi, coerenza, purezza e ARI)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*